# Compound change

What if we change name + pronouns + university all at once? Does the score move by about the sum of the single changes, or more, or less?

In [ ]:
!pip install sentence-transformers pandas scikit-learn

In [ ]:
import os, pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

In [ ]:
from google.colab import files
up = files.upload()
for f in up:
    os.rename(f, f"data/{f}")

In [ ]:
jobs = pd.read_csv("data/jobs.csv")
res = pd.read_csv("data/resume_variants.csv")
jobs["job_text"] = jobs["title"]+" "+jobs["domain"]+" "+jobs["company_name"]+" "+jobs["job_description"]

In [ ]:
# build the "all 3 changed" version by copying values from the existing single-signal CFs
orig = res[res.version=="original"].set_index("resume_id")
nm = res[res.version=="name_changed"].set_index("resume_id")
pn = res[res.version=="pronoun_changed"].set_index("resume_id")
un = res[res.version=="university_changed"].set_index("resume_id")

comp_rows = []
for rid in orig.index:
    o = orig.loc[rid]
    t = str(o["resume_text"])
    t = t.replace(o["name"], nm.loc[rid,"name"])
    t = t.replace(o["pronouns"], pn.loc[rid,"pronouns"])
    t = t.replace(o["university"], un.loc[rid,"university"])
    comp_rows.append({
        "resume_id": rid,
        "domain": o["domain"],
        "version": "all_changed",
        "changed_signal": "name+pronoun+university",
        "name": nm.loc[rid,"name"],
        "pronouns": pn.loc[rid,"pronouns"],
        "university": un.loc[rid,"university"],
        "resume_text": t,
    })
comp = pd.DataFrame(comp_rows)
comp[["resume_id","name","pronouns","university"]]

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
job_emb = model.encode(jobs["job_text"].tolist())
orig_emb = model.encode(orig["resume_text"].tolist())
comp_emb = model.encode(comp["resume_text"].tolist())

In [ ]:
orig_ids = list(orig.index)
comp_ids = list(comp["resume_id"])
out = []
for i, rid in enumerate(orig_ids):
    ci = comp_ids.index(rid)
    for j in range(len(jobs)):
        a = float(cosine_similarity(orig_emb[i:i+1], job_emb[j:j+1])[0][0])
        b = float(cosine_similarity(comp_emb[ci:ci+1], job_emb[j:j+1])[0][0])
        out.append({
            "resume_id": rid,
            "job_id": jobs.iloc[j]["job_id"],
            "job_title": jobs.iloc[j]["title"],
            "original_score": a,
            "compound_score": b,
            "score_difference": b - a,
            "absolute_difference": abs(b - a),
        })
cc = pd.DataFrame(out)
cc.head()

In [ ]:
summary = pd.DataFrame({
    "changed_signal":["name+pronoun+university"],
    "average_score_difference":[cc["score_difference"].mean()],
    "average_absolute_difference":[cc["absolute_difference"].mean()],
    "max_absolute_difference":[cc["absolute_difference"].max()],
})
summary

Compare this to the sum of the three single-signal numbers from fairness_summary.csv. We got compound < sum, so they cancel a bit.

In [ ]:
comp.to_csv("results/compound_resume_variants.csv", index=False)
cc.to_csv("results/compound_comparison.csv", index=False)
summary.to_csv("results/compound_summary.csv", index=False)
for f in ["compound_resume_variants.csv","compound_comparison.csv","compound_summary.csv"]:
    files.download(f"results/{f}")